# Day 3 - Conversational AI - aka Chatbot!

In [ ]:
# importing libraries

import os
from dotenv import load_dotenv
from openai import AzureOpenAI
import gradio as gr
from App.config import azure_endpoint, api_version, headers

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('cd_api_key')

if api_key and api_key.startswith('e4'):
    print('API key found and looks good so far')
else:
    print('API key not found')


In [ ]:
gpt_mini = 'gpt-5-mini' 
gpt_nano = 'gpt-5-nano' 

openai = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version
)

In [ ]:
def call_openai(messages, model):
    response = openai.chat.completions.create(
        messages=messages,
        model=model,
        extra_headers=headers,
    )
    return response.choices[0].message.content


## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [ ]:
def reply_mango(messages, history):
    return 'Mangoes'

In [ ]:
app = gr.ChatInterface(fn=reply_mango)
app.launch()

In [ ]:
def reply_mango_2(messages, history):
    return f'You said {messages}, and the history is {history}, but I still say Mangoes'

In [ ]:
app = gr.ChatInterface(fn=reply_mango_2)
app.launch()

## OK! Let's write a slightly better chat callback!

In [ ]:
system_message = "You are a helpful assistant"

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    new_history = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = call_openai(new_history, gpt_mini)
    return response


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

### Streaming the results

In [ ]:
def stream_chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    new_history = [{'role': 'system', 'content': system_message}] + history + \
                    [{'role': 'user', 'content': message}]
    
    streaming = openai.chat.completions.create(
        messages=new_history,
        model=gpt_mini,
        stream=True,
        extra_headers=headers
    )

    result = ''
    for chunk in streaming:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        result += delta.content or ''
        yield result


In [ ]:
app = gr.ChatInterface(fn=stream_chat)
app.launch()

## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [ ]:
system_prompt = '''
You are a helpful assistant in a clothes store. You should try to gently encourage
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off.
For example, if the customer says 'I'm looking to buy a hat,
You could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.
Encourage the customer to buy hats if they are unsure what to get.
'''

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    new_history = [{'role': 'system', 'content': system_prompt}] + history + \
                    [{'role': 'user', 'content': message}]
    
    streaming = openai.chat.completions.create(
        messages=new_history,
        model=gpt_mini,
        extra_headers=headers,
        stream=True
    )

    result = ''
    for chunk in streaming:
        if not chunk.choices:
            continue 
        delta = chunk.choices[0].delta
        result += delta.content or '' 
        yield result


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

In [ ]:
system_prompt += '''If a customer wants to buy shoes, you should respond that the shoes are not on sale today. \
But remind the customer to look at hats!
'''

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    new_history = [{'role': 'system', 'content': system_prompt}] + history + \
                    [{'role': 'user', 'content': message}]
    
    streaming = openai.chat.completions.create(
        messages=new_history,
        model=gpt_mini,
        extra_headers=headers,
        stream=True
    )

    result = ''
    for chunk in streaming:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        result += delta.content or '' 
        yield result


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    relevant_system_message = system_prompt

    if ('belt' in message.lower()):
        relevant_system_message += f'If the user asks for belts, let them know that the store does not sell belts. Instead ask the customer to try other items that are on sale.'

    new_history = [{'role': 'system', 'content': relevant_system_message}] + history + \
                    [{'role': 'user', 'content': message}]

    streaming = openai.chat.completions.create(
        messages=new_history,
        model=gpt_mini,
        extra_headers=headers,
        stream=True
    )

    result = ''
    for chunk in streaming:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        result += delta.content or ''
        yield result


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>